# Biomarker Dataset - Preprocessing & Unimodal Classifier

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import random
import seaborn as sns
import tensorflow as tf

from imblearn.over_sampling import RandomOverSampler
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

### Biomarker Dataset

#### (does not contain records for all 818 patients in ADNI1)

The biomarker dataset consists of 4 features (**CTWHITE, CTRED, PROTEIN, GLUCOSE**) taken at 5 separate time steps for each patient in ADNI1 (**Baseline, M12, M24, M36, M48**), resulting in 20 collected samples for each patient. The resulting Excel table looks like this:

| Patient ID | CTWHITE | CTRED | PROTEIN | GLUCOSE | CTWHITE_M12 | CTRED_M12 | PROTEIN_M12 | GLUCOSE_M12 | CTWHITE_M24 | CTRED_M24 | PROTEIN_M24 | GLUCOSE_M24 | CTWHITE_M36 | CTRED_M36 | PROTEIN_M36 | GLUCOSE_M36 | CTWHITE_M48 | CTRED_M48 | PROTEIN_M48 | GLUCOSE_M48 |
|------------|---------|-------|---------|---------|-------------|-----------|-------------|-------------|-------------|-----------|-------------|-------------|-------------|-----------|-------------|-------------|-------------|-----------|-------------|-------------|
| 002_S_0295 | 0.0     | 9.0   | 48.0    | 65.0    | 0.0         | 14.0      | 49.0        | 65.0        | NaN         | NaN       | NaN         | NaN         | 1.0         | 1.0       | 50.0        | 73.0        | 2.0         | 1.0       | 56.0        | 79.0        |
| 002_S_0413 | 0.0     | 50.0  | 44.0    | 54.0    | 1.0         | 9.0       | 46.0        | 50.0        | NaN         | NaN       | NaN         | NaN         | NaN         | NaN       | NaN         | NaN         | NaN         | NaN       | NaN         | NaN         |
| 002_S_0559 | 2.0     | 0.0   | 37.0    | 47.0    | 1.0         | 1.0       | 44.0        | 59.0        | 1.0         | 1.0       | 40.0        | 65.0        | 1.0         | 13.0      | 44.0        | 62.0        | NaN         | NaN       | NaN         | NaN         |
| 002_S_0619 | 0.0     | 1.0   | 35.0    | 53.0    | 0.0         | 14.0      | 37.0        | 58.0        | NaN         | NaN       | NaN         | NaN         | NaN         | NaN       | NaN         | NaN         | NaN         | NaN       | NaN         | NaN         |
| 002_S_0685 | 0.0     | 2.0   | 53.0    | 43.0    | 1.0         | 13.0      | 46.0        | 50.0        | NaN         | NaN       | NaN         | NaN         | 1.0         | 2.0       | 52.0        | 53.0        | NaN         | NaN       | NaN         | NaN         |
| ...        | ...     | ...   | ...     | ...     | ...         | ...       | ...         | ...         | ...         | ...       | ...         | ...         | ...         | ...       | ...         | ...         | ...         | ...       | ...         | ...         |

This table represents the DataFrame with the specified columns and data values.

To preprocess this, the code begins by identifying all Excel files within the specified directory. It then initialises an empty DataFrame to aggregate the merged data. The function iterates through each Excel file, reading its contents into a temp DataFrame. 

For each file, it extracts the filename without the extension to use as a suffix for column names to differentiate between timesteps. Using an outer join operation, the function merges the current DataFrame (`df`) with the accumulated dataset (`biomarker_dataset`) based on the shared PTID column. 

The merged DataFrame is then printed, and the count of NaN values in each column is computed and displayed. Finally, the consolidated dataset is saved as a CSV file (BiomarkerDataCSV.csv) without including the index.

In [ ]:
def merge_excel_files(directory):
    # getting list of Excel files in directory
    excel_files = [file for file in os.listdir(directory) if file.endswith('.xlsx') and not file.startswith('~$')]

    # initialising an empty DataFrame to store the merged data
    merged_df = pd.DataFrame()

    # looping through each Excel file
    for file in excel_files:
        # reading Excel file into a DataFrame
        df = pd.read_excel(os.path.join(directory, file))
        
        # extracting filename without extension as suffix
        suffix = os.path.splitext(file)[0]
        
        # merging the current DataFrame with the merged DataFrame
        if merged_df.empty:
            merged_df = df
        else:
            merged_df = pd.merge(merged_df, df, on='PTID', how='outer', suffixes=('', '_' + suffix))

    return merged_df

# directory containing Excel files
directory = r"D:\Biomarkers"

merged_dataframe = merge_excel_files(directory)
print(merged_dataframe)

In [ ]:
# counting NaNs in each column
nan_counts = merged_dataframe.isna().sum()
print("NaN counts per column:")
print(nan_counts)

### Imputing Missing Timestep Sample Values

Some of the patients have missing sample values at certain timesteps, and some do not have any values at all. For the purposes of this research project, instead of dropping the rows with NaN values, imputing will be used to estimate the most likely missing sample values based on the other patient data. 

An Extra Trees Regressor model is instantiated with specific hyperparameters. Then, an IterativeImputer object is created, specifying the estimator as the Extra Trees Regressor model and setting a random state. This imputer will iteratively predict and impute missing values based on other features in the dataset. Next, the columns containing missing values within the DataFrame are identified and stored in the `columns_to_impute` variable.

The IterativeImputer's fit_transform method is applied to the subset of the DataFrame containing only the columns with missing values. This method fits the imputer on the data and then imputes missing values using the fitted model, returning the imputed data. Finally, the missing values in the original DataFrame are replaced with the imputed values, updating the DataFrame in place. The updated DataFrame is then printed to display the imputed values alongside the original data.

This handles missing values in the biomarker dataset, ensuring that imputed values are accurate and feasible for downstream analysis. The iterative imputation approach helps to retain the structure and distribution of the original data while filling in missing values.

In [ ]:
estimator = ExtraTreesRegressor(n_estimators=60, min_samples_split=15, min_samples_leaf=5, random_state=42)

# defining the imputer
imputer = IterativeImputer(estimator=estimator, random_state=42)

# columns to impute
columns_to_impute = merged_dataframe.columns[merged_dataframe.isnull().any()].tolist()

# imputing missing values
imputed_data = imputer.fit_transform(merged_dataframe[columns_to_impute])

# replacing missing values in original DataFrame
merged_dataframe[columns_to_impute] = imputed_data

print(merged_dataframe)

### Scaling Values

To standardise the biomarker data, this step performs standard scaling, a normalisation technique used to rescale features to a range between 0 and 1. This process is beneficial for ensuring that all features contribute equally to the model training process, particularly in algorithms sensitive to the scale of input features.

An instance of the StandardScaler is created and stored in the `scaler` variable. Then, the `fit_transform` method of the scaler is applied to the DataFrame after dropping the `PTID` column. This method fits the scaler to the data and transforms it, resulting in standardised data stored in the `scaled_data` variable. Next, a new DataFrame called `scaled_dataframe` is created using the standardised data, and the columns are labeled accordingly.

Following this, the `PTID` column is extracted from the original DataFrame `merged_dataframe` and stored in the 'patient_id_column' variable. The `scaled_dataframe` is concatenated with the `patient_id_column` to form a new DataFrame called `scaled_dataframe_with_pid`, ensuring that the 'PTID' column is preserved. The `PTID` column in renamed to 'Patient ID'.

By applying scaling to the datasets, each feature's values are proportionally adjusted to a common scale, ensuring a more stable and efficient model training across different datasets and algorithms. 

In [ ]:
scaler = StandardScaler()

# scaling the data
scaled_data = scaler.fit_transform(merged_dataframe.drop(columns=['PTID']))

# converting the scaled data back to a DataFrame
scaled_dataframe = pd.DataFrame(scaled_data, columns=merged_dataframe.drop(columns=['PTID']).columns)

# extracting the 'Patient ID' column from the original merged_dataframe
patient_id_column = merged_dataframe['PTID']

# concatenating the 'Patient ID' column with the scaled dataframe
scaled_dataframe_with_pid = pd.concat([patient_id_column, scaled_dataframe], axis=1)

scaled_dataframe_with_pid.rename(columns={'PTID': 'Patient ID'}, inplace=True)

# displaying the DataFrame after standard scaling with 'Patient ID' column added
print(scaled_dataframe_with_pid)

new_dataframe = scaled_dataframe_with_pid.copy()

In [ ]:
# saving the preprocessed data to PKL
scaled_dataframe_with_pid.to_pickle('BiomarkerProcessedDataCSV')

### Concatenating Diagnosis Labels & Random Oversampling
The next step is to join the biomarker data with the corresponding patient diagnosis label. This is done by processing the clinical dataset and merging it with the existing DataFrame. Then, the code performs random oversampling, and splits the data into train, validation, and test sets.

Random oversampling addresses dataset class imbalance, where one class is significantly underrepresented compared to another. This imbalance can lead to biased models that struggle with minority class learning. In random oversampling, minority class instances are duplicated or synthesized randomly to match the majority class count, balancing the distribution. This helps the model learn from both classes effectively, reducing bias towards the majority class and often improving overall model performance, particularly when the minority class is critical.

Initially, a clinical dataset is read from a CSV file into a DataFrame named `clinical_dataset`. The `Diagnosis` column in the `clinical_dataset` DataFrame is updated to change `Dementia` to `AD` where it occurs. Then, a new DataFrame named `clinical_extracted` is created, containing only the `Patient ID` and `Diagnosis` columns extracted from the `clinical_dataset`.

The `clinical_extracted` DataFrame is merged with the existing DataFrame (`new_dataframe`) based on the `Patient ID` column, resulting in a new DataFrame named `merged_dataframe`. Random oversampling is performed on `merged_dataframe` using the RandomOverSampler from scikit-learn, with the resampled data stored in `X_resampled` and `y_resampled`.

The resampled data is converted back to DataFrames, `X_resampled` and `y_resampled`, and then split into train, validation, and test sets using the train_test_split function from scikit-learn. The shapes of the resulting datasets are printed to confirm the split sizes.

In [ ]:
# reading clinical CSV file into a DataFrame
clinical_dataset = pd.read_csv(r"C:\Users\kishe\Documents\Year 3 Jupyter\ClinicalDatasetCSV")

# replacing 'Dementia' with 'AD' in the 'Diagnosis' column
clinical_dataset['Diagnosis'] = clinical_dataset['Diagnosis'].replace('Dementia', 'AD')

# extracting 'Patient ID' and 'Diagnosis' columns
clinical_extracted = clinical_dataset[['Patient ID', 'Diagnosis']]

# merging with existing DataFrame based on 'Patient ID' column
merged_dataframe = pd.merge(new_dataframe, clinical_extracted, on='Patient ID', how='inner')

# performing random oversampling
oversampler = RandomOverSampler(random_state=42)
X_resampled, y_resampled = oversampler.fit_resample(merged_dataframe.drop(columns=['Patient ID', 'Diagnosis']), merged_dataframe['Diagnosis'])

# converting the resampled data back to DataFrames
X_resampled = pd.DataFrame(X_resampled, columns=merged_dataframe.drop(columns=['Patient ID', 'Diagnosis']).columns)
y_resampled = pd.Series(y_resampled)

# separating X_resampled and y_resampled into train, val, and test sets
X_train, X_temp, y_train, y_temp = train_test_split(X_resampled, y_resampled, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print("X_train shape after oversampling:", X_train.shape)
print("X_val shape after oversampling:", X_val.shape)
print("X_test shape after oversampling:", X_test.shape)
print("y_train shape after oversampling:", y_train.shape)
print("y_val shape after oversampling:", y_val.shape)
print("y_test shape after oversampling:", y_test.shape)

### Preparing the Y Labels
A dictionary (`label_mapping`) is defined to map diagnostic categories ("AD" for Alzheimer's disease, "CN" for cognitively normal, and "MCI" for mild cognitive impairment) to numerical values (0, 1, and 2, respectively). The labels in the training, validation, and test sets are then replaced with their corresponding numerical values using list comprehensions and NumPy arrays.

In [ ]:
# encoding the categorical diagnosis labels with numerical mappings
label_mapping = {"AD": 0, "CN": 1, "MCI": 2}
y_train = np.array([label_mapping[label] for label in y_train])
y_val = np.array([label_mapping[label] for label in y_val])
y_test = np.array([label_mapping[label] for label in y_test])

print(type(y_train))
print(y_train[:20])

### Reshaping into a 3D Array for RNN Input
To prepare the X datasets for the classifier, it converts the DataFrames to numpy arrays and then reshapes them to fit the RNN input format. Each array is reshaped into a 3D array with dimensions (`batch_size, time_steps, features`), where `batch_size` represents the number of samples, `time_steps` represents the number of observations in each sample, and `features` represents the number of variables in each observation. In this case:

`batch_size` = size of the dataset (number of patients)  
`time_steps` = 5: BL, M12, M24, M36, M48  
`features` = 4: **CTWHITE, CTRED, PROTEIN, GLUCOSE**  

This reshaping aligns the data with the RNN's input structure, enabling it to learn temporal patterns within the biomarker data.

In [ ]:
# converting DataFrame to numpy array
X_train_array = X_train.values
X_val_array = X_val.values
X_test_array = X_test.values

# reshape X_train
X_train_rnn = X_train_array.reshape(-1, 5, 4)
print("Shape of X_train_rnn:", X_train_rnn.shape)

# reshape X_val
X_val_rnn = X_val_array.reshape(-1, 5, 4)
print("Shape of X_val_rnn:", X_val_rnn.shape)

# reshape X_test
X_test_rnn = X_test_array.reshape(-1, 5, 4)
print("Shape of X_test_rnn:", X_test_rnn.shape)


### Training the RNN Classifier
Code evaluation loop developed with guidance and adapted from: https://github.com/rsinghlab/MADDi/blob/main/training/train_clinical.py

Initialises an LSTM-based sequential model, compiles it with specified loss and optimisation functions, and trains it using the training data. The model architecture comprises three LSTM layers followed by two densely connected layers with appropriate activation functions. During training, the model evaluates performance metrics such as accuracy and loss on both training and validation datasets. After training, it predicts class probabilities for the test dataset and calculates the confusion matrix, precision, recall, F1-score, and overall accuracy. 

In [ ]:
# setting random seeds for reproducibility
def reset_random_seeds(seed):
    tf.random.set_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

# lists to store evaluation metrics
acc = []
precision = []
recall = []
f1 = []
val_acc = []

# generating random seeds for experiments
seeds = [42, 10, 53, 78, 20]

# looping through each seed
for seed in seeds:
    reset_random_seeds(seed)
    print("Seed:", seed)
    # defining the RNN model
    model = Sequential([
        LSTM(units=256, input_shape=(5, 4), return_sequences=True),
        LSTM(units=128, return_sequences=True),
        LSTM(units=64),
        Dense(16, activation='relu'),
        Dense(3, activation='softmax')
    ])

    # compiling the model
    model.compile(optimizer=Adam(0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    # training the model
    history = model.fit(X_train_rnn, y_train, epochs=64, batch_size=16, validation_data=(X_val_rnn, y_val), verbose=0)

    # evaluating the model on validation data
    val_accuracy = history.history['val_accuracy'][-1]
    print("Validation Accuracy:", val_accuracy)
    val_acc.append(val_accuracy)

    # evaluating the model on test data
    y_probs = model.predict(X_test_rnn)
    y_pred = np.argmax(y_probs, axis=1)
    accuracy = accuracy_score(y_test, y_pred)
    acc.append(accuracy)

    # calculating precision, recall, and f1-score
    cr = classification_report(y_test, y_pred, output_dict=True)
    precision.append(cr["macro avg"]["precision"])
    recall.append(cr["macro avg"]["recall"])
    f1.append(cr["macro avg"]["f1-score"])

# printing average and standard deviation of evaluation metrics
print("Average Validation Accuracy:", np.mean(val_acc))
print("Average Accuracy:", np.mean(acc))
print("Average Precision:", np.mean(precision))
print("Average Recall:", np.mean(recall))
print("Average F1 Score:", np.mean(f1))
print("Standard Deviation Validation Accuracy:", np.std(val_acc))
print("Standard Deviation Accuracy:", np.std(acc))
print("Standard Deviation Precision:", np.std(precision))
print("Standard Deviation Recall:", np.std(recall))
print("Standard Deviation F1 Score:", np.std(f1))
print(acc)
print(precision)
print(recall)
print(f1)
print(val_acc)

## Explainable AI

### Gradient Tape
Gradient Tape is tool for computing gradients of a computation with respect to some inputs. It works by recording operations for automatic differentiation during a forward pass. During the forward pass, TensorFlow records all operations that are executed inside a `GradientTape` context. Then, during the backward pass (when gradients are computed), TensorFlow uses this recorded information to automatically compute the gradients of a specified output with respect to the inputs that were "watched" by the tape. 

For eXplainable AI (XAI) in Recurrent Neural Networks (RNNs), Gradient Tape can be used to understand the importance of each feature or time step in the input sequence with respect to the model's prediction. By watching the input tensor and computing gradients with respect to it, insights are gained into how changes in input features or time steps affect the model's output. This information can be visualised using techniques such as saliency maps, gradient-based class activation maps (Grad-CAM), integrated gradients, or simply by visualising the gradients themselves. These help understand which parts of the input sequence are most influential in driving the model's predictions, thus providing valuable insights into the model's decision-making process.

---

The first code box converts the input data to TensorFlow tensors and then records the operations during the forward pass using GradientTape. Afterward, it computes the gradients of the model's output with respect to the input data. These gradients represent the sensitivity of the model's predictions to changes in input features. The code then calculates the mean importance score for each time step and generates a bar plot to visualise these scores. Each bar in the plot corresponds to a time step, showing the mean importance score of features at that time step. This plot provides insights into which input features are most influential for the RNN model's predictions.

In [ ]:
X_test_rnn_tf = tf.convert_to_tensor(X_test_rnn)
X_train_rnn_tf = tf.convert_to_tensor(X_train_rnn)

# creating a GradientTape
with tf.GradientTape() as tape:
    tape.watch(X_test_rnn_tf)
    output = model(X_test_rnn_tf)

# calculating gradients
gradients = tape.gradient(output, X_test_rnn_tf)

# calculating feature importance scores for each feature at each time step
feature_importance = np.mean(np.abs(gradients), axis=1)

# calculating mean importance scores across time steps for each feature
mean_importance = np.mean(feature_importance, axis=0)

# printing the feature names and their corresponding importance scores
for feature, importance in zip(feature_names, mean_importance):
    print(f"Feature: {feature}, Importance: {importance}")

# defining the feature names in the desired order
feature_names = ['CTWHITE', 'CTRED', 'PROTEIN', 'GLUCOSE']

# plotting feature importance
plt.figure(figsize=(10, 6))
plt.bar(feature_names, mean_importance)
plt.xlabel('Features')
plt.ylabel('Mean Importance Score')
plt.title('Feature Importance for RNN Predictions')
plt.show()

This code box generates a saliency map for the input data using TensorFlow's GradientTape. It computes gradients of the model's output with respect to the input data, normalises them, and then calculates the saliency map by taking the absolute mean of the normalised gradients across features. The resulting map highlights regions of the input data that strongly influence the model's predictions. The map is visualized using matplotlib, with brighter colors indicating higher feature importance.

In [ ]:
# defining the x and y axis labels
x_labels = ['GLUCOSE', 'CTWHITE', 'CTRED', 'PROTEIN']

plt.figure(figsize=(10, 7))
plt.title('Saliency Map')

plt.imshow(saliency_map.numpy(), cmap='jet', interpolation='nearest', aspect='auto')

plt.xticks(ticks=range(len(x_labels)), labels=x_labels, rotation=45)

# adding color bar
plt.colorbar()

plt.show()


Next, this code computes integrated gradients to explain model predictions on input data. Integrated gradients quantify feature importance by integrating gradients of the model's output with respect to inputs along a straight path from a baseline input to the actual input. The function takes the model, input data, optional baseline, and integration steps as inputs. It computes integrated gradients by interpolating between baseline and input, calculating gradients along this path, and averaging them. The resulting integrated gradients are normalised.

In [ ]:
def integrated_gradients(model, inputs, baseline=None, num_steps=50):
    if baseline is None:
        baseline = tf.zeros_like(inputs)
    scaled_inputs = [baseline + (float(i) / num_steps) * (inputs - baseline) for i in range(num_steps + 1)]
    with tf.GradientTape() as tape:
        tape.watch(inputs)
        outputs = model(inputs)
    grads = tape.gradient(outputs, inputs)
    integrated_grads = tf.reduce_mean(grads, axis=0) * (inputs - baseline)
    return integrated_grads

integrated_grads = integrated_gradients(model, X_test_rnn_tf)

# normalising XAI map
integrated_grads_normalized = (integrated_grads - tf.reduce_min(integrated_grads)) / (tf.reduce_max(integrated_grads) - tf.reduce_min(integrated_grads))

# plotting normalised XAI map
plt.figure(figsize=(5, 5))
plt.title('Integrated Gradients')
plt.imshow(integrated_grads_normalized.numpy(), cmap='bone', interpolation='nearest', aspect='auto')  
plt.colorbar()


### Yellowbrick
From: https://www.scikit-yb.org/en/latest/

This code box uses the `FeatureCorrelation` visualiser from the Yellowbrick library to explore the relationship between features and the target variable. It calculates the mutual information between each feature and the target variable for classification tasks. The `feature_names` variable holds the names of the features in the dataset. 

After fitting the visualiser with the training data, it displays the correlation matrix heatmap, where features are sorted based on their mutual information with the target variable. This visualisation helps identify features that are informative for predicting the target.

In [ ]:
from yellowbrick.target import FeatureCorrelation

feature_names = list(X_train.columns)

visualizer = FeatureCorrelation(
    method='mutual_info-classification', feature_names=feature_names, sort=True
)

visualizer.fit(X_train, y_train)     
visualizer.show()      

Next, the `Rank2D` visualiser is used to visualise pairwise feature correlations using Pearson correlation coefficients. Pearson correlation measures the linear relationship between two variables. It provides a value between -1 and 1, where 1 indicates a perfect positive linear relationship, -1 indicates a perfect negative linear relationship, and 0 indicates no linear relationship.

The `Rank2D` visualiser computes Pearson correlation coefficients between all pairs of features in the dataset and displays them in a two-dimensional heatmap. A value closer to 1 or -1 suggests a stronger linear relationship, while values closer to 0 indicate weaker or no linear relationship. This visualisation helps identify potentially redundant or collinear features, providing insights into feature importance and multicollinearity in the dataset.

In [ ]:
from yellowbrick.features import Rank2D

visualizer = Rank2D(algorithm="pearson")
visualizer.fit_transform(X_train)
visualizer.show()